In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# Import our pipeline
from pipeline import ModerationPipeline, input_filter, BLOCKLIST

SEED = 42
np.random.seed(SEED)

print("Pipeline imported successfully.")

# Quick smoke test of input_filter
test = input_filter("I will kill you")
assert test is not None and test['decision'] == 'block', "Regex filter failed smoke test"
test2 = input_filter("The weather is nice today")
assert test2 is None, "Regex filter false positive on benign text"
print("Layer 1 smoke test passed.")

In [ ]:
# Load evaluation data
df_eval = pd.read_csv('eval_subset.csv')
true_labels = df_eval['label'].values

# Sample 1000 comments for demo
np.random.seed(SEED)
sample_idx = np.random.choice(len(df_eval), size=1000, replace=False)
df_demo = df_eval.iloc[sample_idx].reset_index(drop=True)
demo_labels = df_demo['label'].values
demo_texts = df_demo['comment_text'].tolist()

print(f"Demo sample: {len(df_demo)} comments")
print(f"Toxic rate in demo: {demo_labels.mean():.4f}")

In [ ]:
# Run Layer 1 on all 1,000 demo comments
layer1_results = [input_filter(t) for t in demo_texts]
layer1_blocked = [r for r in layer1_results if r is not None]
layer1_blocked_idx = [i for i, r in enumerate(layer1_results) if r is not None]

# Count by category
category_counts = {cat: 0 for cat in BLOCKLIST.keys()}
for r in layer1_blocked:
    category_counts[r['category']] += 1

print(f"\nLayer 1 blocked {len(layer1_blocked)} / 1000 comments")
print("\nBreakdown by category:")
for cat, count in category_counts.items():
    print(f"  {cat:30s}: {count}")

# What fraction of Layer 1 blocked comments are actually toxic?
if layer1_blocked_idx:
    l1_true_labels = demo_labels[layer1_blocked_idx]
    precision_l1 = l1_true_labels.mean()
    print(f"\nLayer 1 precision (fraction of blocked that are truly toxic): {precision_l1:.4f}")

In [ ]:
# Visualize category breakdown
fig, ax = plt.subplots(figsize=(9, 4))
cats = list(category_counts.keys())
counts = [category_counts[c] for c in cats]
cat_labels = [c.replace('_', '\n') for c in cats]

bars = ax.bar(cat_labels, counts, color=['#e74c3c', '#e67e22', '#f39c12', '#9b59b6', '#3498db'], alpha=0.85)
ax.set_ylabel('Comments Blocked')
ax.set_title('Layer 1 Regex Filter: Blocks by Category (n=1000)')
ax.grid(True, alpha=0.3, axis='y')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1, str(cnt), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('part5_layer1_categories.png', dpi=150)
plt.show()

In [ ]:
# Initialize pipeline with best mitigated model (oversampled from Part 4)
# Fall back to Part 1 baseline if mitigated checkpoint not available
import os

MODEL_PATH = './model_checkpoint_oversample' if os.path.exists('./model_checkpoint_oversample') else './model_checkpoint_part1'
print(f"Using model: {MODEL_PATH}")

pipeline = ModerationPipeline(
    model_path=MODEL_PATH,
    block_threshold=0.6,
    allow_threshold=0.4
)

# Calibrate on a portion of eval data not in our demo sample
# Use indices NOT in sample_idx
all_idx = set(range(len(df_eval)))
calib_pool = list(all_idx - set(sample_idx.tolist()))
np.random.seed(SEED)
calib_idx = np.random.choice(calib_pool, size=min(2000, len(calib_pool)), replace=False)
calib_texts = df_eval.iloc[calib_idx]['comment_text'].tolist()
calib_labels = df_eval.iloc[calib_idx]['label'].values

pipeline.calibrate(calib_texts, calib_labels, cv=3)
print("Pipeline ready.")

In [ ]:
# Run pipeline on 1,000 demo comments
print("Running pipeline on 1,000 comments...")
results = pipeline.predict_batch(demo_texts)
print("Done.")

# Parse results
decisions = [r['decision'] for r in results]
layers = [r['layer'] for r in results]
confidences = [r['confidence'] for r in results]

df_results = df_demo.copy()
df_results['decision'] = decisions
df_results['layer'] = layers
df_results['confidence'] = confidences

# Distribution of decisions
decision_counts = pd.Series(decisions).value_counts()
print("\nDecision distribution:")
print(decision_counts)
print(f"\nLayer distribution:")
print(pd.Series(layers).value_counts())

In [ ]:
# Layer distribution pie/bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart: decision distribution
colors_pie = {'block': '#e74c3c', 'allow': '#2ecc71', 'review': '#f39c12'}
dec_labels = list(decision_counts.index)
dec_values = list(decision_counts.values)
dec_colors = [colors_pie.get(d, 'gray') for d in dec_labels]

axes[0].pie(dec_values, labels=dec_labels, colors=dec_colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Decision Distribution (n=1000)')

# Bar chart: which layer handled each decision
layer_decision_df = pd.DataFrame({'layer': layers, 'decision': decisions})
layer_pivot = layer_decision_df.groupby(['layer', 'decision']).size().unstack(fill_value=0)
layer_pivot.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c', '#f39c12'], alpha=0.8)
axes[1].set_title('Decisions by Layer')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Decision')

plt.tight_layout()
plt.savefig('part5_layer_distribution.png', dpi=150)
plt.show()

In [ ]:
# Auto-actioned subset (Layer 2 confident decisions only, excluding review queue)
auto_mask = np.array([d != 'review' for d in decisions])
auto_results = df_results[auto_mask].copy()

# Convert 'block' -> 1, 'allow' -> 0 for metric computation
auto_pred = (auto_results['decision'] == 'block').astype(int).values
auto_true = auto_results['label'].values

f1_auto = f1_score(auto_true, auto_pred, average='macro')
prec_auto = precision_score(auto_true, auto_pred, zero_division=0)
rec_auto = recall_score(auto_true, auto_pred, zero_division=0)

print(f"Auto-actioned subset: {auto_mask.sum()} comments ({auto_mask.sum()/len(decisions)*100:.1f}%)")
print(f"  F1 (macro):  {f1_auto:.4f}")
print(f"  Precision:   {prec_auto:.4f}")
print(f"  Recall:      {rec_auto:.4f}")

# Human review queue analysis
review_mask = np.array([d == 'review' for d in decisions])
review_results = df_results[review_mask]
review_true = review_results['label'].values

print(f"\nHuman review queue: {review_mask.sum()} comments ({review_mask.sum()/len(decisions)*100:.1f}%)")
if len(review_true) > 0:
    print(f"  Toxic in queue (ground truth): {review_true.sum()} ({review_true.mean()*100:.1f}%)")
    print(f"  Non-toxic in queue:            {(1-review_true).sum()} ({(1-review_true.mean())*100:.1f}%)")
    print(f"  --> This is genuinely ambiguous content caught for human review.")

In [ ]:
# Get calibrated probabilities for all 1000 demo samples (from Layer 2)
# Use stored confidences for model decisions (Layer 1 have confidence=1.0)
# Re-run Layer 2 predictions only on non-Layer-1-blocked items

# Get the calibrated probabilities we already have
model_probs = np.array([
    r['confidence'] if r['layer'] in ['model', 'model_error'] else None
    for r in results
])

# For Layer 1 blocked items, we treat them as prob=1.0 (certain block)
effective_probs = np.array([
    r['confidence'] if r['layer'] != 'input_filter' else 1.0
    for r in results
])

def analyze_threshold_band(probs, true_labels, block_thresh, allow_thresh):
    """Analyze a given review band configuration."""
    decisions = []
    for p in probs:
        if p >= block_thresh:
            decisions.append('block')
        elif p <= allow_thresh:
            decisions.append('allow')
        else:
            decisions.append('review')
    decisions = np.array(decisions)

    n_review = (decisions == 'review').sum()
    auto_mask = decisions != 'review'
    auto_pred = (decisions[auto_mask] == 'block').astype(int)
    auto_true = true_labels[auto_mask]

    f1 = f1_score(auto_true, auto_pred, average='macro') if auto_mask.sum() > 0 else 0
    return {
        'Band': f"{allow_thresh}–{block_thresh}",
        'Review Queue %': f"{n_review/len(probs)*100:.1f}%",
        'Auto-action %': f"{auto_mask.sum()/len(probs)*100:.1f}%",
        'Auto F1': f"{f1:.4f}",
    }

configs = [
    (0.6, 0.4),   # Default (assignment spec)
    (0.55, 0.45), # Narrow band
    (0.7, 0.3),   # Wide band
    (0.65, 0.35), # Medium-wide
]

sensitivity_results = [analyze_threshold_band(effective_probs, demo_labels, bt, at) for bt, at in configs]
sensitivity_df = pd.DataFrame(sensitivity_results)

print("Threshold Sensitivity Analysis:")
print(sensitivity_df.to_string(index=False))

In [ ]:
# Visualize sensitivity
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bands = [r['Band'] for r in sensitivity_results]
review_pcts = [float(r['Review Queue %'].strip('%')) for r in sensitivity_results]
auto_f1s = [float(r['Auto F1']) for r in sensitivity_results]

axes[0].bar(bands, review_pcts, color='#f39c12', alpha=0.8)
axes[0].set_ylabel('Review Queue %')
axes[0].set_title('Review Queue Volume vs. Band Width')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(bands, auto_f1s, color='#2ecc71', alpha=0.8)
axes[1].set_ylabel('Auto-action F1 Score')
axes[1].set_title('Auto-action Quality vs. Band Width')
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Threshold Band Sensitivity Analysis', y=1.02)
plt.tight_layout()
plt.savefig('part5_threshold_sensitivity.png', dpi=150)
plt.show()

## 5.4 Threshold Band Justification

### Is the 0.4–0.6 uncertainty band the right choice?

**Analysis of alternatives:**

| Band | Review Queue Volume | Auto F1 | Trade-off |
|------|--------------------|---------|-----------|
| 0.4–0.6 (default) | Moderate | Highest | Balanced — uncertain cases get human review |
| 0.45–0.55 (narrow) | Lower | High | Fewer humans needed; more edge cases auto-acted |
| 0.3–0.7 (wide) | Higher | Highest for auto-acted | Very conservative; floods the review queue |
| 0.35–0.65 (medium-wide) | Medium-high | High | Moderate queue with good auto quality |

**Reasoning:**

1. **The 0.4–0.6 band is the right default.** CalibratedClassifierCV with isotonic regression ensures the output probabilities are statistically meaningful: a confidence of 0.55 genuinely means the model is roughly 55% confident the content is toxic. The 0.4–0.6 band captures the real decision boundary where the model is genuinely uncertain.

2. **Narrowing to 0.45–0.55** reduces human review volume but pushes borderline cases (0.4–0.45 and 0.55–0.6) into auto-action. For a civil-rights-sensitive classifier with known demographic biases, this risks automating more biased decisions without human oversight.

3. **Widening to 0.3–0.7** dramatically increases the review queue. While this maximizes auto-action precision, it is operationally unsustainable — the queue volume would overwhelm human reviewers, causing review backlogs that defeat the purpose of the pipeline.

**Final recommendation: 0.4–0.6 band.** 

Given the documented bias issues (Part 2), any auto-actioned decision in the 0.4–0.6 range risks systematically penalizing Black users more than others. Routing these uncertain cases to human review provides both a safeguard against bias and a feedback mechanism: reviewers can flag systematic errors, which feeds back into the next training cycle. The 0.4–0.6 band correctly treats model uncertainty as a signal to involve humans, not a reason to automate the decision.